In [ ]:
import os
import sys
import pandas as pd
from src.utils.db_utils import get_connection, execute_query

def byeweeks():
    """Bye weeks """
    query = """
    select *
    from stats.byeweek
    where season = '2024
    """
    return  execute_query(query)

bye_weekdf = byeweeks()

print(bye_weekdf)

def get_teamstats_with_game_context(gamesummaryid=None, season=None):
    """Get teamstats with game context fields populated"""
    
    # Base query with the CASE logic
    query = """
    SELECT 
        ts.season,
        ts.week,
        ts.gamesummaryid,
        ts.teamid,
        ts.hometeamid,
        ts.awayteamid,
        ts.total_yards,
        -- Game context fields based on home/away logic
        CASE 
            WHEN ts.teamid = gs.hometeamid THEN gs.days_since_last_game_hometeam
            WHEN ts.teamid = gs.awayteamid THEN gs.days_since_last_game_awayteam
            ELSE NULL
        END as days_since_last_game,
        CASE 
            WHEN ts.teamid = gs.hometeamid THEN gs.hometeam_weeks_from_bye
            WHEN ts.teamid = gs.awayteamid THEN gs.awayteam_weeks_from_bye
            ELSE NULL
        END as weeks_from_bye
    FROM stats.teamstats ts
    LEFT JOIN stats.gamesummary gs ON ts.gamesummaryid = gs.gamesummaryid
    """
    
    # Add WHERE conditions based on parameters
    where_conditions = []
    if gamesummaryid:
        where_conditions.append(f"ts.gamesummaryid = '{gamesummaryid}'")
    if season:
        where_conditions.append(f"ts.season = {season}")
    
    if where_conditions:
        query += " WHERE " + " AND ".join(where_conditions)
    
    query += " ORDER BY ts.gamesummaryid, ts.teamid"
    
    return execute_query(query)

df_test = get_teamstats_with_game_context(gamesummaryid='gs-202410DETHOU')
#print("Test game results:")
#print(df_test)


In [ ]:
## Game Weather data
import os
import sys
import pandas as pd
from src.utils.db_utils import get_connection, execute_query

def gameweather():
    """ Get game weather data """
    query = """
    SELECT 
        gamesummaryid, season, week, hometeamid, awayteamid, 
        kickoff_temperature_f, kickoff_wind_speed_mph, kickoff_wind_direction_deg, kickoff_humidity_pct,kickoff_pressure_mb, kickoff_precipitation_in, kickoff_weather_description,
        game_total_precipitation_in, game_total_rain_in, game_total_snow_in, 
        game_avg_pressure_mb, game_avg_wind_speed_mph, game_avg_wind_direction_deg

    From stats.gameweather
    WHERE season = '2024'          
    """
    return execute_query(query)

game_weather_df = gameweather()
print(game_weather_df)

Successfully connected to the database!
       gamesummaryid  season  week hometeamid awayteamid  \
0    gs-202401BALKAN    2024     1        KAN        BAL   
1    gs-202401GNBPHI    2024     1        PHI        GNB   
2    gs-202401ARIBUF    2024     1        BUF        ARI   
3    gs-202401HOUIND    2024     1        IND        HOU   
4    gs-202401CARNOR    2024     1        NOR        CAR   
..               ...     ...   ...        ...        ...   
280  gs-202420LARPHI    2024    20        PHI        LAR   
281  gs-202420BALBUF    2024    20        BUF        BAL   
282  gs-202421WASPHI    2024    21        PHI        WAS   
283  gs-202421BUFKAN    2024    21        KAN        BUF   
284  gs-202422KANPHI    2024    22        PHI        KAN   

     kickoff_temperature_f  kickoff_wind_speed_mph  \
0                     76.3                     4.2   
1                     61.6                     6.0   
2                     58.9                    18.3   
3                     6